In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

import joblib
import json
import warnings

warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


In [5]:
print(pd.__version__)

3.0.2


In [6]:
import pandas as pd

In [11]:
pip install xlrd

   ---------------------------------------- 0.0/96.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/96.6 kB ? eta -:--:--
   ---- ----------------------------------- 10.2/96.6 kB ? eta -:--:--
   ---- ----------------------------------- 10.2/96.6 kB ? eta -:--:--
   ---- ----------------------------------- 10.2/96.6 kB ? eta -:--:--
   ------------ --------------------------- 30.7/96.6 kB 119.1 kB/s eta 0:00:01
   ------------ --------------------------- 30.7/96.6 kB 119.1 kB/s eta 0:00:01
   ------------ --------------------------- 30.7/96.6 kB 119.1 kB/s eta 0:00:01
   ------------ --------------------------- 30.7/96.6 kB 119.1 kB/s eta 0:00:01
   ------------ --------------------------- 30.7/96.6 kB 119.1 kB/s eta 0:00:01
   ---------------------------------------- 96.6/96.6 kB 204.5 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
df = pd.read_excel('CTG.xls')

print(df.head())
print(df.shape)

  Unnamed: 0  Unnamed: 1                             Cardiotocographic data  \
0        NaN         NaN  2126 measurements and classifications of foeta...   
1        NaN         NaN                                                NaN   
2  Worksheet         NaN                                           Raw Data   
3        NaN         NaN                                                NaN   
4        NaN         NaN                                                NaN   

  Unnamed: 3  Unnamed: 4  Unnamed: 5  Unnamed: 6  Unnamed: 7  Unnamed: 8  \
0        NaN         NaN         NaN         NaN         NaN         NaN   
1        NaN         NaN         NaN         NaN         NaN         NaN   
2        NaN         NaN         NaN         NaN         NaN         NaN   
3        NaN         NaN         NaN         NaN         NaN         NaN   
4        NaN         NaN         NaN         NaN         NaN         NaN   

                               Unnamed: 9  Unnamed: 10 Unnamed: 11  

In [13]:
df = pd.read_excel('CTG.xls', sheet_name='Raw Data')

print(df.head())
print(df.shape)

       FileName       Date      SegFile      b       e    LBE     LB   AC  \
0           NaN        NaT          NaN    NaN     NaN    NaN    NaN  NaN   
1  Variab10.txt 1996-12-01  CTG0001.txt  240.0   357.0  120.0  120.0  0.0   
2    Fmcs_1.txt 1996-05-03  CTG0002.txt    5.0   632.0  132.0  132.0  4.0   
3    Fmcs_1.txt 1996-05-03  CTG0003.txt  177.0   779.0  133.0  133.0  2.0   
4    Fmcs_1.txt 1996-05-03  CTG0004.txt  411.0  1192.0  134.0  134.0  2.0   

    FM   UC  ...    C    D    E   AD   DE   LD   FS  SUSP  CLASS  NSP  
0  NaN  NaN  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN   NaN    NaN  NaN  
1  0.0  0.0  ...  0.0  0.0  0.0  0.0  0.0  0.0  1.0   0.0    9.0  2.0  
2  0.0  4.0  ...  0.0  0.0  0.0  1.0  0.0  0.0  0.0   0.0    6.0  1.0  
3  0.0  5.0  ...  0.0  0.0  0.0  1.0  0.0  0.0  0.0   0.0    6.0  1.0  
4  0.0  6.0  ...  0.0  0.0  0.0  1.0  0.0  0.0  0.0   0.0    6.0  1.0  

[5 rows x 40 columns]
(2130, 40)


In [14]:
# Remove completely empty rows
df = df.dropna(how='all')

# Remove rows where NSP is missing
df = df.dropna(subset=['NSP'])

# Check shape again
print(df.shape)

# Check missing values
print(df.isnull().sum())

(2126, 40)
FileName    0
Date        0
SegFile     0
b           0
e           0
LBE         0
LB          0
AC          0
FM          0
UC          0
ASTV        0
MSTV        0
ALTV        0
MLTV        0
DL          0
DS          0
DP          0
DR          0
Width       0
Min         0
Max         0
Nmax        0
Nzeros      0
Mode        0
Mean        0
Median      0
Variance    0
Tendency    0
A           0
B           0
C           0
D           0
E           0
AD          0
DE          0
LD          0
FS          0
SUSP        0
CLASS       0
NSP         0
dtype: int64


In [15]:
# Select important numeric features
features = [
    'LB', 'AC', 'FM', 'UC', 'ASTV', 'MSTV',
    'ALTV', 'MLTV', 'Width', 'Min',
    'Max', 'Nmax', 'Nzeros', 'Mode',
    'Mean', 'Median', 'Variance', 'Tendency'
]

# Target column
target = 'NSP'

# Create X and y
X = df[features]
y = df[target]

print(X.head())
print(y.head())

      LB   AC   FM   UC  ASTV  MSTV  ALTV  MLTV  Width   Min    Max  Nmax  \
1  120.0  0.0  0.0  0.0  73.0   0.5  43.0   2.4   64.0  62.0  126.0   2.0   
2  132.0  4.0  0.0  4.0  17.0   2.1   0.0  10.4  130.0  68.0  198.0   6.0   
3  133.0  2.0  0.0  5.0  16.0   2.1   0.0  13.4  130.0  68.0  198.0   5.0   
4  134.0  2.0  0.0  6.0  16.0   2.4   0.0  23.0  117.0  53.0  170.0  11.0   
5  132.0  4.0  0.0  5.0  16.0   2.4   0.0  19.9  117.0  53.0  170.0   9.0   

   Nzeros   Mode   Mean  Median  Variance  Tendency  
1     0.0  120.0  137.0   121.0      73.0       1.0  
2     1.0  141.0  136.0   140.0      12.0       0.0  
3     1.0  141.0  135.0   138.0      13.0       0.0  
4     0.0  137.0  134.0   137.0      13.0       1.0  
5     0.0  137.0  136.0   138.0      11.0       1.0  
1    2.0
2    1.0
3    1.0
4    1.0
5    1.0
Name: NSP, dtype: float64


In [16]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1700, 18)
(426, 18)


In [17]:
# Scale the data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete")

Scaling complete


In [18]:
# Logistic Regression
lr_model = LogisticRegression()
lr_model.fit(X_train_scaled, y_train)

# KNN
knn_model = KNeighborsClassifier()
knn_model.fit(X_train_scaled, y_train)

# Random Forest
rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)

print("All models trained successfully")

All models trained successfully


In [19]:
# Model scores
print("Logistic Regression:", lr_model.score(X_test_scaled, y_test))
print("KNN:", knn_model.score(X_test_scaled, y_test))
print("Random Forest:", rf_model.score(X_test, y_test))

Logistic Regression: 0.8661971830985915
KNN: 0.9061032863849765
Random Forest: 0.9483568075117371


In [20]:
# Predictions using Random Forest
y_pred = rf_model.predict(X_test)

# Classification report
report = classification_report(y_test, y_pred, output_dict=True)

# Print report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         1.0       0.97      0.98      0.97       333
         2.0       0.88      0.81      0.85        64
         3.0       0.87      0.93      0.90        29

    accuracy                           0.95       426
   macro avg       0.91      0.91      0.91       426
weighted avg       0.95      0.95      0.95       426



In [21]:
# Save classification report
with open('classification_report.json', 'w') as f:
    json.dump(report, f)

print("Classification report saved")

Classification report saved


In [22]:
# Save model
joblib.dump(rf_model, 'rf_best_model.pkl')

print("Model saved successfully")

Model saved successfully


In [23]:
# Load model
loaded_model = joblib.load('rf_best_model.pkl')

# Test prediction
sample_prediction = loaded_model.predict(X_test.iloc[:5])

print(sample_prediction)

[1. 1. 1. 1. 1.]
